
`from docutils.nodes import description` 这行代码来自于 Python 的 **Docutils** 库。

以下是关于它的详细介绍：

### 1. 什么是 `docutils`？
**Docutils** 是 Python 生态系统中处理纯文本及其结构化的核心库。它最著名的用途是处理 **reStructuredText (reST)** 格式（文件后缀通常为 `.rst`）。

*   它是 Python 官方文档生成的基石。
*   它是著名的文档生成工具 **Sphinx** 的基础。
*   它用于将纯文本转换为 HTML、LaTeX、XML 等格式。

### 2. `docutils.nodes` 是什么？
`nodes` 模块定义了 Docutils 的 **文档树（Document Tree）** 结构。

当你解析一个文档时，Docutils 不会直接把它变成 HTML，而是先把它解析成一个由各种“节点（Nodes）”组成的树状结构（类似于浏览器里的 DOM 树或编程中的 AST 抽象语法树）。

在这个模块里，有各种类代表文档的不同部分，例如：
*   `paragraph`（段落）
*   `title`（标题）
*   `list`（列表）
*   `image`（图片）
*   **`description`**（描述）

### 3. `description` 类具体是做什么的？

`description` 是 `docutils.nodes` 中的一个特定节点类。它通常用于表示文档的 **元数据（Metadata）** 或 **书目字段（Bibliographic Fields）** 中的“描述”信息。

#### 场景示例：
在 reStructuredText (`.rst`) 文件开头，通常会包含文档信息（DocInfo）：

```rst
:Author: 张三
:Version: 1.0
:Description: 这是一个关于人工智能的文档，
              详细介绍了 LLM 的原理。
```

当 Docutils 解析上述文本时：
1.  它会识别出 `:Description:` 这个字段。
2.  它会创建一个 `docutils.nodes.description` 类的实例。
3.  这个实例包含了“这是一个关于人工智能的文档...”这段文本。

### 4. 为什么你会看到这个依赖？

如果你在阅读 LangChain 或其他处理文档加载（Document Loader）的源码时看到了这个包，通常是因为：

1.  **解析 .rst 文件**：该库正在试图加载 reStructuredText 格式的文档，并将其转换为大模型可以理解的文本。
2.  **提取元数据**：代码可能正在遍历文档树，专门寻找 `description` 节点，以便将文档的摘要提取出来，作为 LangChain `Document` 对象的 `metadata`。
3.  **UnstructuredLoader**：很多非结构化文档处理库（如 `unstructured`）底层也会依赖 `docutils` 来处理特定格式的文本。

### 总结
*   **库**: Docutils (处理 reST 格式的标准库)
*   **模块**: `nodes` (定义文档结构节点)
*   **类**: `description` (专门用于存储文档元数据中的“描述”信息的节点)

# 1、使用@tool装饰器定义工具


使用 @tool 装饰器是 LangChain 中创建自定义工具最简单、最常用的方法。它能自动将 Python 函数转换为 Agent 可以理解的工具格式（包括名称、描述和参数结构）。



1. 基础用法：最简单的工具定义

这是最直接的方式。LangChain 会自动提取：
- 工具名称：默认为函数名。
- 工具描述：默认为函数的文档字符串（Docstring）。
- 参数结构：根据函数的类型提示（Type Hints）生成。

In [2]:
from langchain_core.tools import tool, StructuredTool


@tool
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #默认是函数的名称
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")  #默认是函数的说明信息
print(f"return_direct = {add_number.return_direct}")  #默认值是False


# 1. 定义函数并加上 @tool 装饰器
@tool
def multiply(a: int, b: int) -> int:
    """
    计算两个整数相乘的结果。
    当你需要进行数学乘法运算时使用此工具。
    """
    return a * b

# 2. 查看生成的工具信息
print(f"工具名称: {multiply.name}")
print(f"工具描述: {multiply.description}")
print(f"参数结构: {multiply.args}")

# 3. 调用工具 (注意：调用时通常传入字典)
result = multiply.invoke({"a": 5, "b": 3})
print(f"运行结果: {result}")

name = add_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = 计算两个整数的和
return_direct = False
工具名称: multiply
工具描述: 计算两个整数相乘的结果。
当你需要进行数学乘法运算时使用此工具。
参数结构: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
运行结果: 15


关键点：
类型提示 (: int) 是必须的，否则 Agent 不知道怎么传参。
文档字符串 ("""...""") 非常重要，Agent 靠它决定是否使用这个工具。

## 2. 进阶用法：自定义名称和行为

有时候函数名称不适合做工具名，或者你想跳过LLM的最终处理，可以在装饰器中传入参数

In [3]:
@tool("search_google", return_direct=True)
def search(query: str) -> str:
    """
    这是一个模拟的搜索引擎。
    当用户需要查询实时信息或不知道的知识时使用。
    """
    return f"搜索结果: '{query}' 的相关信息如下..."

print(f"工具名称: {search.name}")          # search_google
print(f"直接返回: {search.return_direct}") # True

"""
第一个参数：自定义工具名称（覆盖函数名）。
return_direct=True：
如果为 False（默认）：工具的输出会传回给 LLM，LLM 总结后再回复用户。
如果为 True：工具的输出直接作为最终结果展示给用户（中断 Agent 思考链）。
"""

工具名称: search_google
直接返回: True


## 3. 高级用法：使用 Pydantic 精确控制参数

当你的工具参数较多，或者需要给每个参数提供详细说明（Description）以帮助 LLM 理解时，建议结合 Pydantic 使用 args_schema。

为什么这么做？
如果只有 a: int，LLM 知道是整数，但不知道 a 代表什么。

通过 Pydantic，你可以告诉 LLM a 是“股票代码”，b 是“天数”。



In [4]:
from langchain_core.tools import tool
from pydantic import BaseModel,Field

# 1. 定义参数模型（Schema）
class StockQueryInput(BaseModel):
    symbol: str = Field(description="股票代码，例如 'AAPL' 或 'TSLA'")
    days: int = Field(description="查询过去多少天的数据，默认是 1")

# 2. 将 Schema 绑定到工具
@tool("get_stock_price",args_schema=StockQueryInput)
def get_stock_price(symbol: str, days: int = 1) -> str:
    """
    查询指定股票在过去几天内的价格信息
    """
    return f"股票: {symbol} 在过去 {days} 天的平均价格是 $150。"

# 查看详细的参数列表
print(get_stock_price.args)

{'symbol': {'description': "股票代码，例如 'AAPL' 或 'TSLA'", 'title': 'Symbol', 'type': 'string'}, 'days': {'description': '查询过去多少天的数据，默认是 1', 'title': 'Days', 'type': 'integer'}}


## 4. 实际场景演示：错误处理

在写工具时，抛出异常也是一种与 LLM 交互的方式。

如果工具报错，LangChain 会捕捉异常并告诉 LLM "出错了，请重试"，LLM 往往会尝试修正参数再次调用。


In [5]:
def divide(a: int, b: int) -> float:
    """计算两个数相除"""
    if b == 0:
        # 抛出异常 Agent 会收到错误信息并可能尝试自我修正
        raise ValueError("除数不能为 0，请检查参数 b 的值。")
    return a / b

try:
    print(divide.invoke({"a": 10, "b": 0}))
except Exception as e:
    # 在实际 Agent 运行中，这个错误会被自动捕获并反馈给 LLM
    print(f"工具运行出错: {e}")

工具运行出错: 'function' object has no attribute 'invoke'


## 总结

使用 @tool 定义工具的最佳实践：
1. Docstring (文档)：必须写清楚工具是做什么的以及什么时候用。
2. Type Hints (类型)：必须为所有参数添加类型注解。
3. Args Description (参数描述)：对于复杂参数，尽量使用 Pydantic 的 Field(description="...") 来描述参数的含义，这能显著提高 Agent 调用的成功率。

举例2：


In [2]:
from langchain_core.tools import tool


@tool(name_or_callable="add_two_number", description="add two numbers", return_direct=True)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #add_two_number
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")  #add two numbers
print(f"return_direct = {add_number.return_direct}")  #True

name = add_two_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = add two numbers
return_direct = True


In [3]:
#调用工具
add_number.invoke({"a": 10, "b": 20})


30

举例3：修改args参数的描述

In [5]:
from pydantic import Field
from langchain_core.tools import tool
from pydantic import BaseModel


class FieldInfo(BaseModel):
    a: int = Field(description="第1个整型参数")
    b: int = Field(description="第2个整型参数")


@tool(name_or_callable="add_two_number", description="add two numbers", return_direct=True, args_schema=FieldInfo)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #add_two_number
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")  #add two numbers
print(f"return_direct = {add_number.return_direct}")  #True

name = add_two_number
args = {'a': {'description': '第1个整型参数', 'title': 'A', 'type': 'integer'}, 'b': {'description': '第2个整型参数', 'title': 'B', 'type': 'integer'}}
description = add two numbers
return_direct = True


# 2、StructuredTool的from_function()的使用


StructuredTool.from_function() 是 LangChain 中创建工具的另一种核心方法。

虽然 @tool 装饰器非常方便，但在某些场景下（例如你无法修改原函数的代码，或者需要动态生成工具时），StructuredTool.from_function() 更加灵活和强大。

它允许你通过编程的方式，将任何现有的 Python 函数包装成 LangChain 工具。

## 1.核心参数

在使用之前，先了解一下它的的主要参数：


举例1：

In [6]:
from langchain_core.tools.structured import StructuredTool


# 声明一个函数
def search_google(query: str):
    return "最后查询的结果"


# 定义一个工具
search01 = StructuredTool.from_function(
    func=search_google,
    name="Search",
    description="查询google搜索引擎，并将结果返回"
)

print(f"name = {search01.name}")
print(f"args = {search01.args}")
print(f"description = {search01.description}")
print(f"return_direct = {search01.return_direct}")

name = Search
args = {'query': {'title': 'Query', 'type': 'string'}}
description = 查询google搜索引擎，并将结果返回
return_direct = False


In [5]:
search01.invoke({"query":"中美AI的发展现状"})

'最后查询的结果'

举例2：


In [11]:
from langchain_core.tools.structured import StructuredTool
from pydantic import BaseModel,Field

class FieldInfo(BaseModel):
    query: str = Field(description="要检索的关键词")


# 声明一个函数
def search_google(query: str):
    return "最后查询的结果"


# 定义一个工具
search02 = StructuredTool.from_function(
    func=search_google,
    name="Search",
    description="查询google搜索引擎，并将结果返回",
    return_direct=True,
    args_schema=FieldInfo
)

print(f"name = {search02.name}")
print(f"args = {search02.args}")
print(f"description = {search02.description}")
print(f"return_direct = {search02.return_direct}")

name = Search
args = {'query': {'description': '要检索的关键词', 'title': 'Query', 'type': 'string'}}
description = 查询google搜索引擎，并将结果返回
return_direct = True
